# RAW -> BRONZE PIPELINE

In this pipeline we are going lo load the data from the INPUT FOLDER to a table in the BRONZE layer.

In [0]:
import datetime

# 2️⃣ Rutas
input_folder = "/Workspace/Users/atriguerosholgado@gmail.com/RETAIL-SALES/DATA/INPUT/"
historical_folder = "/Workspace/Users/atriguerosholgado@gmail.com/RETAIL-SALES/DATA/HISTORICAL DATA/"

# 3️⃣ Listar archivos CSV en INPUT
files = dbutils.fs.ls(input_folder)
csv_files = [f.path for f in files if f.path.endswith(".csv")]

# 4️⃣ Comprobación: solo 1 CSV
if len(csv_files) != 1:
    raise Exception(f"❌ Error: Se esperaba un único CSV, pero se encontraron {len(csv_files)}: {csv_files}")

csv_path = csv_files[0]
print(f"✅ CSV encontrado: {csv_path}")

INSERT THE DATA IN retail_sales.bronze.raw_sales

In [0]:
spark.sql(f"""
INSERT INTO retail_sales.bronze.raw_sales
SELECT
  `Transaction ID`       AS Transaction_ID,
  TO_DATE(Date, 'yyyy-MM-dd') AS Date_t,
  `Customer ID`          AS Customer_ID,
  Gender,
  Age,
  `Product Category`     AS Product_Category,
  Quantity,
  `Price per Unit`       AS Price_per_Unit,
  `Total Amount`         AS Total_Amount
FROM read_files(
  '{csv_path}',
  format => 'csv',
  header => 'true',
  inferSchema => 'true'
)
""")

In [0]:
filename = csv_path.split("/")[-1]
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
historical_path = historical_folder + filename.replace(".csv", f"_{timestamp}.csv")

# Copiar y eliminar original (efecto mv)
dbutils.fs.cp(csv_path, historical_path)
dbutils.fs.rm(csv_path)

print(f"✅ CSV movido a HISTORICAL DATA: {historical_path}")